# ListT5 Grouping Method Experiment

Small inference-only experiment for the ListT5 paper (`2402.15838v3.pdf`). The paper uses ListT5 with an m-ary tournament sort: candidates are split into sequential groups of `listwise_k=5`, ListT5 selects the winners from each group, and the tournament repeats until the top reranked documents are found.

This notebook keeps the official ListT5 inference and BEIR evaluation code unchanged, and only swaps the grouping phase used inside tournament sort.

Main comparison:
- `sequential`: official contiguous chunking from `run_listt5.py`.
- `score_balanced`: sorts candidates by first-stage rank and distributes them round-robin across groups.
- `random`: seeded random grouping for a sanity-check baseline.

No training is performed. The expected baseline check is Table 2 ListT5-base, BM25 top-100, `r=2`, NDCG@10.

## 1. Setup

Run this notebook from the `Project` directory. It expects the local repo at `Project/ListT5`. A CUDA GPU is strongly recommended because the official evaluator moves the model to `cuda`.

In [1]:
# Run this cell before the imports below.
# Kaggle usually already includes torch/transformers. Avoid pinning transformers==4.33.3 here
# because its old tokenizers dependency may fail to build on newer Python kernels.
!pip install -q jsonlines sentencepiece huggingface_hub beir

In [2]:
import sys
!git clone https://github.com/soyoung97/ListT5.git
sys.path.append("/kaggle/working/ListT5")

fatal: destination path 'ListT5' already exists and is not an empty directory.


In [3]:
from pathlib import Path
import json
import math
import os
import queue
import random
import subprocess
import sys
import threading
import time
from types import SimpleNamespace

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "ListT5").exists() and (PROJECT_ROOT.parent / "ListT5").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

LISTT5_ROOT = PROJECT_ROOT / "ListT5"
if not (LISTT5_ROOT / "run_listt5.py").exists() and (LISTT5_ROOT / "ListT5" / "run_listt5.py").exists():
    LISTT5_ROOT = LISTT5_ROOT / "ListT5"
assert (LISTT5_ROOT / "run_listt5.py").exists(), f"Could not find run_listt5.py under {PROJECT_ROOT}"

sys.path.insert(0, str(LISTT5_ROOT))

import torch
from beir_eval import run_rerank_eval
from beir_length_mapping import BEIR_LENGTH_MAPPING
import FiDT5 as fid_module
from run_listt5 import ListT5Evaluator, read_jsonl

# Compatibility patch for newer transformers versions.
# Recent transformers checks tied weights at encoder.embed_tokens, while ListT5 wraps
# the encoder inside EncoderWrapper. Expose the wrapped embedding module there.
if not hasattr(fid_module.EncoderWrapper, "embed_tokens"):
    fid_module.EncoderWrapper.embed_tokens = property(lambda self: self.encoder.embed_tokens)

# Newer transformers passes more positional arguments into T5 blocks than
# the original ListT5 CheckpointWrapper.forward accepted. For inference we
# do not need checkpointing, so forward all args directly to the wrapped block.
def _checkpoint_wrapper_forward_compat(self, *args, **kwargs):
    if self.use_checkpoint and self.training:
        return torch.utils.checkpoint.checkpoint(lambda *inner_args: self.module(*inner_args, **kwargs), *args)
    return self.module(*args, **kwargs)

fid_module.CheckpointWrapper.forward = _checkpoint_wrapper_forward_compat

DATA_DIR = PROJECT_ROOT / "data" / "beir-eval-bm25-top100"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "grouping_method_experiment"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LIVE_RESULTS_CSV = OUTPUT_DIR / "results_live.csv"
LIVE_RESULTS_TXT = OUTPUT_DIR / "results_live.txt"
BASELINE_SUMMARY_CSV = OUTPUT_DIR / "baseline_check.csv"
BASELINE_SUMMARY_TXT = OUTPUT_DIR / "baseline_check.txt"
COMPARISON_SUMMARY_CSV = OUTPUT_DIR / "grouping_comparison.csv"
COMPARISON_SUMMARY_TXT = OUTPUT_DIR / "grouping_comparison.txt"

print("Project root:", PROJECT_ROOT)
print("ListT5 root:", LISTT5_ROOT)
print("Data dir:", DATA_DIR)
print("Output dir:", OUTPUT_DIR)
print("Live results CSV:", LIVE_RESULTS_CSV)

/usr/local/lib/python3.12/dist-packages/beir/util.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


Project root: /kaggle/working
ListT5 root: /kaggle/working/ListT5
Data dir: /kaggle/working/data/beir-eval-bm25-top100
Output dir: /kaggle/working/outputs/grouping_method_experiment
Live results CSV: /kaggle/working/outputs/grouping_method_experiment/results_live.csv


In [4]:
# Kaggle self-contained helper: write the two-GPU worker script automatically.
# This cell makes GPU_MODE='parallel_2gpu' work even when only the notebook is uploaded.
WORKER_SOURCE = r'''import argparse
import json
import math
import os
import random
import sys
import time
from pathlib import Path
from types import SimpleNamespace

os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("OMP_NUM_THREADS", "1")

import torch


def sequential_groups(items, group_size: int, seed: int = 0):
    items = list(items)
    return [items[i:i + group_size] for i in range(0, len(items), group_size)]


def score_balanced_groups(items, group_size: int, seed: int = 0):
    items = list(items)
    if len(items) <= group_size:
        return [items]

    ranked = sorted(items)
    n_groups = math.ceil(len(ranked) / group_size)
    groups = [[] for _ in range(n_groups)]
    for i, item in enumerate(ranked):
        groups[i % n_groups].append(item)
    return [group for group in groups if group]


def random_groups(items, group_size: int, seed: int = 0):
    items = list(items)
    rng = random.Random(seed)
    rng.shuffle(items)
    return sequential_groups(items, group_size, seed=seed)


GROUPING_POLICIES = {
    "sequential": sequential_groups,
    "score_balanced": score_balanced_groups,
    "random": random_groups,
}


def patch_fidt5(fid_module):
    if not hasattr(fid_module.EncoderWrapper, "embed_tokens"):
        fid_module.EncoderWrapper.embed_tokens = property(lambda self: self.encoder.embed_tokens)

    def checkpoint_wrapper_forward_compat(self, *args, **kwargs):
        if self.use_checkpoint and self.training:
            return torch.utils.checkpoint.checkpoint(
                lambda *inner_args: self.module(*inner_args, **kwargs),
                *args,
            )
        return self.module(*args, **kwargs)

    fid_module.CheckpointWrapper.forward = checkpoint_wrapper_forward_compat


def build_evaluator_class(listt5_evaluator_cls, fid_module):
    class GroupingListT5Evaluator(listt5_evaluator_cls):
        def load_model(self):
            start = time.time()
            print("Loading model..", flush=True)
            print(f"Loading fid model from {self.args.model_path}", flush=True)
            model = fid_module.FiDT5.from_pretrained(
                self.args.model_path,
                use_safetensors=False,
            ).to("cuda")
            model.eval()
            print(f"Done! took {time.time() - start} second", flush=True)
            return model

        def group2chunks(self, l, n=5):
            strategy = getattr(self.args, "grouping_strategy", "sequential")
            seed = getattr(self.args, "seed", 0)
            try:
                group_fn = GROUPING_POLICIES[strategy]
            except KeyError as exc:
                raise ValueError(f"Unknown grouping strategy: {strategy}") from exc
            yield from group_fn(l, n, seed=seed)

        def run_inference(self, input_tensors):
            output = self.model.generate(
                **input_tensors,
                max_length=self.args.max_gen_length,
                return_dict_in_generate=True,
                output_scores=True,
            )
            self.num_forward += 1
            print_every = getattr(self.args, "print_every_forwards", 20)
            if print_every and self.num_forward % print_every == 0:
                print(f"[progress] forward_calls={self.num_forward}", flush=True)
            return output

    return GroupingListT5Evaluator


def output_is_complete(output_path: Path, input_path: str, read_jsonl):
    if not output_path.exists():
        return False
    try:
        return len(read_jsonl(str(output_path))) == len(read_jsonl(input_path))
    except Exception:
        return False


def safe_float(value):
    try:
        return float(value)
    except Exception:
        return None


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", required=True)
    args = parser.parse_args()

    with open(args.config, "r", encoding="utf-8") as f:
        config = json.load(f)

    listt5_root = Path(config["listt5_root"])
    sys.path.insert(0, str(listt5_root))

    import FiDT5 as fid_module
    from beir_eval import run_rerank_eval
    from run_listt5 import ListT5Evaluator, read_jsonl

    patch_fidt5(fid_module)
    evaluator_cls = build_evaluator_class(ListT5Evaluator, fid_module)

    job_args = SimpleNamespace(**config["args"])
    output_path = Path(job_args.output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    dataset = config["dataset"]
    strategy = config["strategy"]
    seed = config["seed"]
    gpu_id = os.environ.get("CUDA_VISIBLE_DEVICES", "default")

    print(
        f"[job start] gpu={gpu_id} dataset={dataset} strategy={strategy} seed={seed}",
        flush=True,
    )
    start = time.time()

    if output_is_complete(output_path, job_args.input_path, read_jsonl):
        ndcg10, metric_text = run_rerank_eval(str(output_path))
        mode = "reused_output"
    else:
        evaluator = evaluator_cls(job_args)
        ndcg10, metric_text = evaluator.run_tournament_sort()
        mode = "new_inference"

    elapsed = time.time() - start
    table2 = config["table2"].get(dataset)
    ndcg_float = safe_float(ndcg10)
    row = {
        "dataset": dataset,
        "strategy": strategy,
        "seed": seed,
        "max_queries": config["max_queries"],
        "ndcg@10": ndcg_float,
        "table2_listt5_base_r2": table2,
        "delta_vs_table2": None
        if table2 is None or config["max_queries"] is not None or ndcg_float is None
        else ndcg_float - table2,
        "seconds": elapsed,
        "mode": mode,
        "output_path": str(output_path),
    }

    result_path = Path(config["result_path"])
    result_path.parent.mkdir(parents=True, exist_ok=True)
    with open(result_path, "w", encoding="utf-8") as f:
        json.dump(row, f, indent=2)

    print(
        f"[job done] gpu={gpu_id} dataset={dataset} strategy={strategy} "
        f"ndcg@10={row['ndcg@10']} seconds={elapsed:.1f}",
        flush=True,
    )


if __name__ == "__main__":
    main()
'''

WORKER_SCRIPT_CANDIDATES = [
    PROJECT_ROOT / "ListT5" / "notebook" / "listt5_grouping_worker.py",
    PROJECT_ROOT / "notebook" / "listt5_grouping_worker.py",
    LISTT5_ROOT.parent / "notebook" / "listt5_grouping_worker.py",
]

written_worker_paths = []
seen_worker_paths = set()
for worker_script in WORKER_SCRIPT_CANDIDATES:
    worker_script = worker_script.resolve()
    if worker_script in seen_worker_paths:
        continue
    seen_worker_paths.add(worker_script)
    worker_script.parent.mkdir(parents=True, exist_ok=True)
    worker_script.write_text(WORKER_SOURCE, encoding="utf-8")
    written_worker_paths.append(worker_script)

WORKER_SCRIPT_PATH = written_worker_paths[0]
print("Worker script auto-written to:", WORKER_SCRIPT_PATH)


Worker script auto-written to: /kaggle/working/ListT5/notebook/listt5_grouping_worker.py


## 2. Experiment Knobs

Defaults are chosen to match Table 2 for ListT5-base on BM25 top-100: `listwise_k=5`, `out_k=2`, `topk=100`, `rerank_topk=10`. The dataset list starts with five representative BEIR datasets and can be expanded by adding names from `TABLE2_LISTT5_BASE_TOP100`.

Set `GPU_MODE = "single"` for the original one-GPU path. Set `GPU_MODE = "parallel_2gpu"` to run independent dataset/strategy jobs on `GPU_IDS = ["0", "1"]`. Section 7 writes live results after every completed approach to `outputs/grouping_method_experiment/results_live.csv` and `.txt`.

In [5]:
MODEL_PATH = "Soyoung97/ListT5-base"
HF_DATASET_REPO = "Soyoung97/beir-eval-bm25-top100"

# Set to None for full Table 2-style evaluation. Use a small number, e.g. 10, for a smoke test.
MAX_QUERIES = None

DEFAULT_DATASETS = [
    "trec-covid",
    "nfcorpus",
    "fiqa",
    "scifact",
    "arguana",
]

STRATEGIES = ["sequential", "score_balanced", "random"]
SEEDS = [0]  # Add more seeds for random grouping, e.g. [0, 1, 2].

LISTWISE_K = 5
OUT_K = 2
TOPK = 100
RERANK_TOPK = 10
BATCH_SIZE = 20

# GPU_MODE options:
# - "single": current behavior, one run at a time on the active CUDA device.
# - "parallel_2gpu": run independent jobs in subprocesses, one per GPU id below.
# If the two-GPU mode causes issues, switch this back to "single" and rerun Section 7.
GPU_MODE = "parallel_2gpu"
GPU_IDS = ["0", "1"]

# Plain print progress for Kaggle Save & Run logs, where tqdm may not render well.
PRINT_EVERY_FORWARDS = 20

# Paper Table 2, BM25 top-100, ListT5-base (r=2), NDCG@10.
TABLE2_LISTT5_BASE_TOP100 = {
    "trec-covid": 0.783,
    "nfcorpus": 0.356,
    "bioasq": 0.564,
    "nq": 0.531,
    "hotpotqa": 0.726,
    "fiqa": 0.396,
    "signal": 0.335,
    "news": 0.485,
    "robust04": 0.521,
    "arguana": 0.489,
    "touche": 0.334,
    "cqadupstack": 0.388,
    "quora": 0.864,
    "dbpedia-entity": 0.437,
    "scidocs": 0.176,
    "fever": 0.798,
    "climate-fever": 0.240,
    "scifact": 0.741,
}

pd.DataFrame(
    [{"dataset": d, "table2_listt5_base_r2_ndcg10": TABLE2_LISTT5_BASE_TOP100[d]} for d in DEFAULT_DATASETS]
)

,dataset,table2_listt5_base_r2_ndcg10
0,trec-covid,0.783
1,nfcorpus,0.356
2,fiqa,0.396
3,scifact,0.741
4,arguana,0.489


## 3. Data Helper

The official README uses `Soyoung97/beir-eval-bm25-top100`. This helper reuses a local file if available. The repo already includes `ListT5/trec-covid.jsonl`, so that dataset can run without downloading. Other datasets are downloaded through `huggingface_hub` on first use.

In [6]:
def write_jsonl(path, rows):
    import jsonlines
    with jsonlines.open(path, "w") as writer:
        writer.write_all(rows)


def dataset_path(dataset_name: str, max_queries: int | None = None) -> Path:
    """Return a local JSONL path for a BEIR top-100 dataset."""
    local_copy = DATA_DIR / f"{dataset_name}.jsonl"
    bundled_copy = LISTT5_ROOT / f"{dataset_name}.jsonl"

    if local_copy.exists():
        base_path = local_copy
    elif bundled_copy.exists():
        base_path = bundled_copy
    else:
        try:
            from huggingface_hub import hf_hub_download
        except ImportError as exc:
            raise ImportError("Install huggingface_hub to download BEIR JSONL files.") from exc

        downloaded = hf_hub_download(
            repo_id=HF_DATASET_REPO,
            filename=f"{dataset_name}.jsonl",
            repo_type="dataset",
            local_dir=str(DATA_DIR),
            local_dir_use_symlinks=False,
        )
        base_path = Path(downloaded)

    if max_queries is None:
        return base_path

    subset_path = DATA_DIR / f"{dataset_name}.first{max_queries}.jsonl"
    if not subset_path.exists():
        rows = read_jsonl(str(base_path))[:max_queries]
        write_jsonl(subset_path, rows)
    return subset_path


for dname in DEFAULT_DATASETS:
    print(dname, "->", dataset_path(dname, MAX_QUERIES))

trec-covid -> /kaggle/working/ListT5/trec-covid.jsonl
nfcorpus -> /kaggle/working/data/beir-eval-bm25-top100/nfcorpus.jsonl
fiqa -> /kaggle/working/data/beir-eval-bm25-top100/fiqa.jsonl
scifact -> /kaggle/working/data/beir-eval-bm25-top100/scifact.jsonl
arguana -> /kaggle/working/data/beir-eval-bm25-top100/arguana.jsonl


## 4. Grouping Policies

`sequential_groups` is intentionally identical to the official `group2chunks` behavior: contiguous groups of size `listwise_k`. The alternatives below return the same kind of list-of-index-groups, so the rest of `run_listt5.py` stays untouched.

In [7]:
def sequential_groups(items, group_size: int, seed: int = 0):
    items = list(items)
    return [items[i:i + group_size] for i in range(0, len(items), group_size)]


def score_balanced_groups(items, group_size: int, seed: int = 0):
    """Spread high and low first-stage ranks across groups.

    Candidate index 0 is the first-stage rank 1 document, index 1 is rank 2, etc.
    Sorting by index reconstructs the first-stage order, then round-robin assignment
    prevents one group from containing only the strongest initial candidates.
    """
    items = list(items)
    if len(items) <= group_size:
        return [items]

    ranked = sorted(items)
    n_groups = math.ceil(len(ranked) / group_size)
    groups = [[] for _ in range(n_groups)]
    for i, item in enumerate(ranked):
        groups[i % n_groups].append(item)
    return [group for group in groups if group]


def random_groups(items, group_size: int, seed: int = 0):
    items = list(items)
    rng = random.Random(seed)
    rng.shuffle(items)
    return sequential_groups(items, group_size, seed=seed)


GROUPING_POLICIES = {
    "sequential": sequential_groups,
    "score_balanced": score_balanced_groups,
    "random": random_groups,
}

demo = list(range(20))
pd.concat(
    [
        pd.DataFrame({"strategy": name, "group": i + 1, "indices": [group]})
        for name, fn in GROUPING_POLICIES.items()
        for i, group in enumerate(fn(demo, LISTWISE_K, seed=0))
    ],
    ignore_index=True,
)

,strategy,group,indices
0,sequential,1,"[0, 1, 2, 3, 4]"
1,sequential,2,"[5, 6, 7, 8, 9]"
2,sequential,3,"[10, 11, 12, 13, 14]"
3,sequential,4,"[15, 16, 17, 18, 19]"
4,score_balanced,1,"[0, 4, 8, 12, 16]"
5,score_balanced,2,"[1, 5, 9, 13, 17]"
6,score_balanced,3,"[2, 6, 10, 14, 18]"
7,score_balanced,4,"[3, 7, 11, 15, 19]"
8,random,1,"[10, 18, 16, 14, 0]"
9,random,2,"[17, 11, 2, 3, 9]"


## 5. Evaluator Subclass

This is the only integration point. `GroupingListT5Evaluator` inherits the official evaluator and overrides `group2chunks`. Model loading, generation, output scoring, caching, and BEIR metrics are still handled by the original code.

In [8]:
class GroupingListT5Evaluator(ListT5Evaluator):
    def load_model(self):
        start = time.time()
        print("Loading model..")
        print(f"Loading fid model from {self.args.model_path}")
        model = fid_module.FiDT5.from_pretrained(self.args.model_path, use_safetensors=False).to("cuda")
        model.eval()
        print(f"Done! took {time.time() - start} second")
        return model

    def group2chunks(self, l, n=5):
        strategy = getattr(self.args, "grouping_strategy", "sequential")
        seed = getattr(self.args, "seed", 0)
        try:
            group_fn = GROUPING_POLICIES[strategy]
        except KeyError as exc:
            raise ValueError(f"Unknown grouping strategy: {strategy}") from exc
        yield from group_fn(l, n, seed=seed)

    def run_inference(self, input_tensors):
        output = self.model.generate(**input_tensors,
                max_length=self.args.max_gen_length,
                return_dict_in_generate=True, output_scores=True)
        self.num_forward += 1
        print_every = getattr(self.args, "print_every_forwards", PRINT_EVERY_FORWARDS)
        if print_every and self.num_forward % print_every == 0:
            print(f"[progress] forward_calls={self.num_forward}", flush=True)
        return output


def make_args(dataset_name: str, strategy: str, seed: int = 0, max_queries: int | None = None):
    input_path = dataset_path(dataset_name, max_queries=max_queries)
    subset_tag = "full" if max_queries is None else f"first{max_queries}"
    output_path = OUTPUT_DIR / strategy / f"seed{seed}" / subset_tag / f"{dataset_name}_output.jsonl"
    output_path.parent.mkdir(parents=True, exist_ok=True)

    max_input_length = BEIR_LENGTH_MAPPING.get(dataset_name)
    if max_input_length is None:
        raise ValueError(f"No max input length for dataset {dataset_name}. Add it to BEIR_LENGTH_MAPPING or pass a known BEIR name.")

    return SimpleNamespace(
        firststage_result_key="bm25_results",
        docid_key="docid",
        pid_key="pid",
        qrels_key="qrels",
        score_key="bm25_score",
        question_text_key="q_text",
        text_key="text",
        title_key="title",
        model_path=MODEL_PATH,
        topk=TOPK,
        max_input_length=max_input_length,
        padding="max_length",
        listwise_k=LISTWISE_K,
        rerank_topk=RERANK_TOPK,
        out_k=OUT_K,
        dummy_number=21,
        verbose=False,
        seed=seed,
        bsize=BATCH_SIZE,
        input_path=str(input_path),
        output_path=str(output_path),
        measure_flops=False,
        skip_no_candidate=False,
        skip_issubset=False,
        max_gen_length=LISTWISE_K + 2,
        grouping_strategy=strategy,
        print_every_forwards=PRINT_EVERY_FORWARDS,
    )

## 6. Run One Dataset

Start with one dataset and sequential grouping. If this is a full run, the NDCG@10 should be close to the Table 2 reference for the same dataset. Small differences can happen from dependency or hardware differences; subset runs should not be compared directly to Table 2.

In [9]:
def output_is_complete(output_path: Path, input_path: str) -> bool:
    if not output_path.exists():
        return False
    try:
        return len(read_jsonl(str(output_path))) == len(read_jsonl(input_path))
    except Exception:
        return False


def save_results_snapshot(rows, csv_path: Path = LIVE_RESULTS_CSV, txt_path: Path = LIVE_RESULTS_TXT):
    """Persist the current experiment table after each completed approach."""
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    key_cols = ["dataset", "strategy", "seed", "max_queries"]
    df = df.drop_duplicates(subset=key_cols, keep="last")
    df = df.sort_values(key_cols).reset_index(drop=True)
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(csv_path, index=False)
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(df.to_string(index=False))
        f.write("\n")
    print(f"[saved] live results -> {csv_path}", flush=True)
    print(f"[saved] live results -> {txt_path}", flush=True)
    return df


def run_single(dataset_name: str, strategy: str, seed: int = 0, max_queries: int | None = None, reuse_existing: bool = True):
    args = make_args(dataset_name, strategy, seed=seed, max_queries=max_queries)
    output_path = Path(args.output_path)
    start = time.time()
    print(f"[job start] dataset={dataset_name} strategy={strategy} seed={seed} output={output_path}", flush=True)

    if reuse_existing and output_is_complete(output_path, args.input_path):
        print(f"[reuse] Found complete output at {output_path}", flush=True)
        ndcg10, metric_text = run_rerank_eval(str(output_path))
        mode = "reused_output"
    else:
        evaluator = GroupingListT5Evaluator(args)
        ndcg10, metric_text = evaluator.run_tournament_sort()
        mode = "new_inference"

    elapsed = time.time() - start
    table2 = TABLE2_LISTT5_BASE_TOP100.get(dataset_name)
    print(f"[job done] dataset={dataset_name} strategy={strategy} seed={seed} ndcg@10={ndcg10} seconds={elapsed:.1f}", flush=True)
    return {
        "dataset": dataset_name,
        "strategy": strategy,
        "seed": seed,
        "max_queries": max_queries,
        "ndcg@10": float(ndcg10),
        "table2_listt5_base_r2": table2,
        "delta_vs_table2": None if table2 is None or max_queries is not None else float(ndcg10) - table2,
        "seconds": elapsed,
        "mode": mode,
        "output_path": str(output_path),
    }


# Smoke check: uncomment to run one small subset quickly.
# run_single("trec-covid", "sequential", max_queries=5, reuse_existing=True)

## 7. Run the Main Comparison

This cell runs each grouping method for the configured datasets. For Table 2 matching, keep `MAX_QUERIES = None` and include `sequential` in `STRATEGIES`.

Progress is printed with plain `print(...)` messages for Kaggle Save & Run logs. After each completed approach, the current result table is saved to `results_live.csv` and `results_live.txt`.

In [10]:
def iter_experiment_jobs(datasets=DEFAULT_DATASETS, strategies=STRATEGIES, seeds=SEEDS):
    for dataset_name in datasets:
        for strategy in strategies:
            strategy_seeds = seeds if strategy == "random" else [seeds[0]]
            for seed in strategy_seeds:
                yield {"dataset": dataset_name, "strategy": strategy, "seed": seed}


def run_grid_single(datasets=DEFAULT_DATASETS, strategies=STRATEGIES, seeds=SEEDS, max_queries=MAX_QUERIES):
    rows = []
    jobs = list(iter_experiment_jobs(datasets, strategies, seeds))
    print(f"[grid] mode=single total_jobs={len(jobs)}", flush=True)
    for job_idx, job in enumerate(jobs, start=1):
        dataset_name = job["dataset"]
        strategy = job["strategy"]
        seed = job["seed"]
        print(f"\n=== job {job_idx}/{len(jobs)} | {dataset_name} | {strategy} | seed={seed} ===", flush=True)
        rows.append(run_single(dataset_name, strategy, seed=seed, max_queries=max_queries))
        save_results_snapshot(rows)
        display(pd.DataFrame(rows).tail(1))
    return save_results_snapshot(rows)


def find_worker_script():
    candidates = [
        PROJECT_ROOT / "ListT5" / "notebook" / "listt5_grouping_worker.py",
        PROJECT_ROOT / "notebook" / "listt5_grouping_worker.py",
        LISTT5_ROOT.parent / "notebook" / "listt5_grouping_worker.py",
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError("Could not find listt5_grouping_worker.py. Use GPU_MODE='single' or place the worker next to this notebook.")


def write_parallel_config(job, gpu_id, max_queries):
    args = make_args(job["dataset"], job["strategy"], seed=job["seed"], max_queries=max_queries)
    config_dir = OUTPUT_DIR / "parallel_job_configs"
    result_dir = OUTPUT_DIR / "parallel_job_results"
    config_dir.mkdir(parents=True, exist_ok=True)
    result_dir.mkdir(parents=True, exist_ok=True)
    subset_tag = "full" if max_queries is None else f"first{max_queries}"
    stem = f"{job['dataset']}__{job['strategy']}__seed{job['seed']}__{subset_tag}"
    result_path = result_dir / f"{stem}.json"
    config = {
        "project_root": str(PROJECT_ROOT),
        "listt5_root": str(LISTT5_ROOT),
        "dataset": job["dataset"],
        "strategy": job["strategy"],
        "seed": job["seed"],
        "max_queries": max_queries,
        "table2": TABLE2_LISTT5_BASE_TOP100,
        "result_path": str(result_path),
        "args": vars(args),
    }
    config_path = config_dir / f"{stem}.json"
    with open(config_path, "w", encoding="utf-8") as f:
        json.dump(config, f, indent=2)
    return args, config_path, result_path


def run_subprocess_job_on_gpu(job, gpu_id, max_queries):
    args, config_path, result_path = write_parallel_config(job, gpu_id, max_queries)
    output_path = Path(args.output_path)
    if output_is_complete(output_path, args.input_path):
        print(f"[gpu {gpu_id}] [reuse] {job['dataset']} | {job['strategy']} | seed={job['seed']}", flush=True)
        return run_single(job["dataset"], job["strategy"], seed=job["seed"], max_queries=max_queries, reuse_existing=True)

    worker_script = find_worker_script()
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
    env["PYTHONUNBUFFERED"] = "1"
    command = [sys.executable, str(worker_script), "--config", str(config_path)]
    print(f"[gpu {gpu_id}] launching {job['dataset']} | {job['strategy']} | seed={job['seed']}", flush=True)
    proc = subprocess.Popen(
        command,
        cwd=str(PROJECT_ROOT),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(f"[gpu {gpu_id}] {line}", end="", flush=True)
    return_code = proc.wait()
    if return_code != 0:
        raise RuntimeError(f"Worker failed on gpu {gpu_id} with return code {return_code}: {job}")
    with open(result_path, "r", encoding="utf-8") as f:
        return json.load(f)


def run_grid_parallel_2gpu(datasets=DEFAULT_DATASETS, strategies=STRATEGIES, seeds=SEEDS, max_queries=MAX_QUERIES, gpu_ids=GPU_IDS):
    jobs = list(iter_experiment_jobs(datasets, strategies, seeds))
    job_queue = queue.Queue()
    for job in jobs:
        job_queue.put(job)

    rows = []
    errors = []
    lock = threading.Lock()
    print(f"[grid] mode=parallel_2gpu total_jobs={len(jobs)} gpu_ids={gpu_ids}", flush=True)

    def gpu_worker(gpu_id):
        while True:
            try:
                job = job_queue.get_nowait()
            except queue.Empty:
                return
            try:
                row = run_subprocess_job_on_gpu(job, gpu_id, max_queries)
                with lock:
                    rows.append(row)
                    save_results_snapshot(rows)
                    print(f"[grid] completed {len(rows)}/{len(jobs)} jobs", flush=True)
                    print(pd.DataFrame([row]).to_string(index=False), flush=True)
            except Exception as exc:
                with lock:
                    errors.append((job, repr(exc)))
                    print(f"[grid error] gpu={gpu_id} job={job} error={exc!r}", flush=True)
            finally:
                job_queue.task_done()

    threads = [threading.Thread(target=gpu_worker, args=(gpu_id,), daemon=True) for gpu_id in gpu_ids]
    for thread in threads:
        thread.start()
    for thread in threads:
        thread.join()

    if errors:
        raise RuntimeError(f"{len(errors)} parallel jobs failed. First error: {errors[0]}")
    return save_results_snapshot(rows).sort_values(["dataset", "strategy", "seed"]).reset_index(drop=True)


def run_grid(datasets=DEFAULT_DATASETS, strategies=STRATEGIES, seeds=SEEDS, max_queries=MAX_QUERIES, gpu_mode=GPU_MODE):
    if gpu_mode == "single":
        return run_grid_single(datasets, strategies, seeds, max_queries)
    if gpu_mode == "parallel_2gpu":
        return run_grid_parallel_2gpu(datasets, strategies, seeds, max_queries, gpu_ids=GPU_IDS)
    raise ValueError(f"Unknown GPU_MODE: {gpu_mode}")


# This is the main experiment call.
results = run_grid()
results

[grid] mode=parallel_2gpu total_jobs=15 gpu_ids=['0', '1']
[gpu 0] launching trec-covid | sequential | seed=0
[gpu 1] launching trec-covid | score_balanced | seed=0
[gpu 0] [job start] gpu=0 dataset=trec-covid strategy=sequential seed=0
[gpu 1] [job start] gpu=1 dataset=trec-covid strategy=score_balanced seed=0
[gpu 1] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
[gpu 0] Input path: /kaggle/working/ListT5/trec-covid.jsonl
[gpu 0] Loading model..
[gpu 0] Loading fid model from Soyoung97/ListT5-base
[gpu 1] Input path: /kaggle/working/ListT5/trec-covid.jsonl
[gpu 1] Loading model..
[gpu 1] Loading fid model from Soyoung97/ListT5-base
[gpu 0] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
[gpu 0] 
[gpu 0] Loading weights:  26%|██▌       | 67/260 [00:00<00:00, 749.96it/s, Materializing param=decoder.block.5.l

KeyboardInterrupt: 

## 8. Summary Tables

The first table checks whether official sequential grouping reproduces the paper baseline. The second table compares grouping methods directly on NDCG@10.

In [ ]:
baseline_check = (
    results[results["strategy"] == "sequential"]
    [["dataset", "ndcg@10", "table2_listt5_base_r2", "delta_vs_table2", "output_path"]]
    .sort_values("dataset")
)
baseline_check.to_csv(BASELINE_SUMMARY_CSV, index=False)
with open(BASELINE_SUMMARY_TXT, "w", encoding="utf-8") as f:
    f.write(baseline_check.to_string(index=False))
    f.write("\n")
print(f"[saved] baseline summary -> {BASELINE_SUMMARY_CSV}", flush=True)
print(f"[saved] baseline summary -> {BASELINE_SUMMARY_TXT}", flush=True)
baseline_check

In [ ]:
comparison = results.pivot_table(
    index="dataset",
    columns="strategy",
    values="ndcg@10",
    aggfunc="mean",
)

if "sequential" in comparison.columns:
    for col in comparison.columns:
        comparison[f"{col}_minus_sequential"] = comparison[col] - comparison["sequential"]

comparison_table = comparison.reset_index()
comparison_table.to_csv(COMPARISON_SUMMARY_CSV, index=False)
with open(COMPARISON_SUMMARY_TXT, "w", encoding="utf-8") as f:
    f.write(comparison_table.to_string(index=False))
    f.write("\n")
print(f"[saved] grouping comparison -> {COMPARISON_SUMMARY_CSV}", flush=True)
print(f"[saved] grouping comparison -> {COMPARISON_SUMMARY_TXT}", flush=True)
comparison_table

## 9. Optional: Expand to More Datasets

To expand beyond the first five datasets, set `DEFAULT_DATASETS` to any subset of the keys below and rerun Sections 7 and 8. Keep `MAX_QUERIES = None` for full comparable numbers.

In [ ]:
pd.DataFrame(
    [{"dataset": name, "table2_listt5_base_r2_ndcg10": score, "max_input_length": BEIR_LENGTH_MAPPING.get(name)}
     for name, score in TABLE2_LISTT5_BASE_TOP100.items()]
).sort_values("dataset")

[gpu 0]   5%|▌         | 1/20 [00:11<03:30, 11.05s/it]
[gpu 0] 
[gpu 0]  78%|███████▊  | 39/50 [02:08<00:36,  3.29s/it]
[gpu 0] Traceback (most recent call last):
[gpu 0]   File "/kaggle/working/ListT5/notebook/listt5_grouping_worker.py", line 191, in <module>
[gpu 0]     main()
[gpu 0]   File "/kaggle/working/ListT5/notebook/listt5_grouping_worker.py", line 157, in main
[gpu 0]     ndcg10, metric_text = evaluator.run_tournament_sort()
[gpu 0]                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
[gpu 0]   File "/kaggle/working/ListT5/run_listt5.py", line 322, in run_tournament_sort
[gpu 0]     output = self.run_batchwise_caching(batch_holder)
[gpu 0]              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
[gpu 0]   File "/kaggle/working/ListT5/run_listt5.py", line 259, in run_batchwise_caching
[gpu 0]     output = self.run_inference({'input_ids': batch_inputids, 'attention_mask': batch_attnmasks})
[gpu 0]              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

## 10. Notes for Reporting

- The independent variable is only `grouping_strategy`; all other ListT5 inference parameters stay fixed.
- Use the sequential full-run row as the reproducibility check against Table 2.
- Report `score_balanced - sequential` per dataset to make the grouping effect visible.
- Treat `random` as a sanity-check baseline. If using random grouping, run multiple seeds and report mean and standard deviation.
- Subset runs are useful for debugging but should not be used as Table 2 replication evidence.